# QLoRA fine-tuning for code refinement — Colab / cloud GPU runner

Runs the **exact same pipeline** as the repo, on a GPU. Nothing here is a
notebook-only reimplementation: every cell shells out to the same `coderefine`
CLI you run locally, so results are reproducible outside Colab.

**Runtime → Change runtime type → T4 GPU** (or better) before running.

| GPU | VRAM | Mistral-7B QLoRA, 2k examples, 3 epochs |
|---|---|---|
| T4 (free tier) | 16 GB | ~50–70 min |
| L4 | 24 GB | ~25–35 min |
| A100 | 40 GB | ~10–15 min |

In [1]:
# 0 — check the GPU we were given
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


## 1 — Get the code and the data

Two options. **A** clones your repo (put your URL in). **B** uploads a zip of
the project folder — use it if the repo is private.

The raw `Code_Refinement/*.jsonl` dumps are ~6 GB, which is too large to upload
comfortably. Either mount Drive with them, or — much faster — copy up only the
**already-curated** `data/` directory (a few MB), which is all training needs.

In [11]:
# 1A — clone the repo (curated data is committed, so no raw-dump download needed)
REPO_URL = "https://github.com/hamnaraeel/lora-code-refinement.git"
!rm -rf lora-code-refinement
!git clone -q $REPO_URL
%cd lora-code-refinement
!ls -la data/processed data/benchmark

Skipping clone — using direct zip upload in the next cell.


In [12]:
# 1B — not needed this run: 1A already cloned everything, curated data included.
print("Skipping — data/processed and data/benchmark came with the git clone above.")

Select coderefine_project.zip when prompted...


KeyboardInterrupt: 

In [ ]:
# 1C — not needed this run (no raw dumps to fetch; curated splits are already in the repo)

## 2 — Install

`bitsandbytes` is what makes 4-bit QLoRA possible and only exists for CUDA,
which is exactly why the local Intel-Mac path in this repo cannot run it.

In [ ]:
# 2 — Install
#
# Pinned to the exact combo verified end-to-end on this project (real training
# run completed, 144 tests passing): transformers 4.46.3 / peft 0.13.2 /
# trl 0.11.4 / accelerate 1.2.1. This matters more than it looks: current
# Colab images ship a much newer TRL that replaced the completion-only-loss
# collator with a chat-template-marker mechanism TRL only auto-patches for a
# small allowlist of models (Mistral is not on it), which fails outright
# rather than silently mis-masking anything. Pinning below avoids that entirely
# by using the older, thoroughly-tested masking path.
!pip install -q "transformers==4.46.3" "peft==0.13.2" "trl==0.11.4" "accelerate==1.2.1" "tokenizers<0.21" \
    "bitsandbytes>=0.43.1" "datasets>=2.19" sacrebleu Levenshtein pyyaml pydantic typer rich matplotlib wandb
!pip install -q -e . --no-deps

import sys
if "/content/lora-code-refinement/src" not in sys.path:
    sys.path.insert(0, "/content/lora-code-refinement/src")

import torch, transformers, peft, trl, accelerate, bitsandbytes
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0), "| vram_gb:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__, "| accelerate", accelerate.__version__)
print("bitsandbytes", bitsandbytes.__version__)

from trl import DataCollatorForCompletionOnlyLM  # must succeed — confirms the legacy masking path is active
print("legacy masking path: OK")

## 3 — Credentials

* **Hugging Face** — needed for gated bases (Llama 3). Mistral-7B-Instruct-v0.3 is ungated.
* **Weights & Biases** — optional. Without a key the pipeline still logs
  everything to `artifacts/runs/<name>/metrics.jsonl`; it degrades, it does not fail.

In [ ]:
import os, getpass

# from huggingface_hub import login; login()   # only needed for gated models

use_wandb = False  # set True to log to W&B
if use_wandb:
    os.environ["WANDB_API_KEY"] = getpass.getpass("W&B API key: ")
    os.environ["WANDB_PROJECT"] = "lora-code-refinement"

## 4 — Build the dataset

Skip this if you copied a prepared `data/` directory up. It needs the raw
`Code_Refinement/*.jsonl` dumps present.

In [ ]:
import pathlib
if pathlib.Path("Code_Refinement/ref-train.jsonl").exists():
    !coderefine build-data --n-train 2000 --n-valid 250 --n-test 250
    !coderefine build-benchmark
else:
    print("Raw dumps absent — assuming data/ was copied up.")
!ls -la data/processed data/benchmark

## 5 — Train

One command, one config file. Everything that affects the result lives in the
YAML, so this run is reproducible anywhere.

In [ ]:
!coderefine train configs/qlora_mistral7b.yaml

In [ ]:
# Loss curves from the local metric log (works with or without W&B)
import json, pathlib
import matplotlib.pyplot as plt

run = "qlora-mistral7b-r16"
rows = [json.loads(l) for l in (pathlib.Path("artifacts/runs")/run/"metrics.jsonl").read_text().splitlines() if l.strip()]
tr = [(r["_step"], r["loss"]) for r in rows if "loss" in r]
ev = [(r["_step"], r["eval_loss"]) for r in rows if "eval_loss" in r]

fig, ax = plt.subplots(figsize=(8, 4.5))
if tr: ax.plot(*zip(*tr), label="train loss", lw=1.4)
if ev: ax.plot(*zip(*ev), label="validation loss", marker="o", lw=1.6)
if ev:
    bs, bl = min(ev, key=lambda x: x[1])
    ax.axvline(bs, ls="--", c="gray", lw=1)
    ax.annotate(f"best checkpoint\nstep {bs} · {bl:.4f}", (bs, bl),
                textcoords="offset points", xytext=(10, 18), fontsize=9)
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_title(f"{run} — training curves")
ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
fig.savefig("reports/training_curves.png", dpi=150)
plt.show()

## 6 — Hyperparameter sweep

Eight arms varying rank (8/16/32), learning rate (1e-4/2e-4/5e-4), epochs
(1/3/5) and target modules (q,v vs all-linear). On a T4 this is several hours —
run a subset first if you are on the free tier and may be pre-empted.

In [ ]:
# Subset (fast): rank sweep only
!bash scripts/run_sweep.sh configs/sweep/s01_r8_lr2e4_e3.yaml configs/sweep/s02_r16_lr2e4_e3.yaml configs/sweep/s03_r32_lr2e4_e3.yaml

# Everything:
# !bash scripts/run_sweep.sh

## 7 — Evaluate

Base first — it is the denominator of every improvement claim — then the tuned
model on the identical benchmark with the identical prompts and greedy decoding.

In [ ]:
ADAPTER = "artifacts/runs/qlora-mistral7b-r16/adapter"
BASE    = "mistralai/Mistral-7B-Instruct-v0.3"

!coderefine evaluate --split benchmark --base-model $BASE --tag base --load-in-4bit
!coderefine evaluate --split benchmark --base-model $BASE --adapter $ADAPTER --tag tuned --load-in-4bit
!coderefine compare artifacts/eval/base__benchmark.predictions.jsonl \
                    artifacts/eval/tuned__benchmark.predictions.jsonl

In [ ]:
# LLM-as-judge (optional — costs API credits)
# import os, getpass
# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
# !coderefine judge artifacts/eval/base__benchmark.predictions.jsonl \
#                   artifacts/eval/tuned__benchmark.predictions.jsonl --provider anthropic

In [ ]:
# Catastrophic forgetting: general capability, before and after
!coderefine forgetting --base-model $BASE --load-in-4bit
!coderefine forgetting --base-model $BASE --adapter $ADAPTER --load-in-4bit

## 8 — The sacred test split

Only run this once, after the configuration is frozen. Everything above used
validation and the benchmark.

In [ ]:
!coderefine evaluate --split test --final --base-model $BASE --tag base-test  --load-in-4bit
!coderefine evaluate --split test --final --base-model $BASE --adapter $ADAPTER --tag tuned-test --load-in-4bit
!coderefine compare artifacts/eval/base-test__test.predictions.jsonl \
                    artifacts/eval/tuned-test__test.predictions.jsonl \
                    --out-path artifacts/eval/comparison_test.json

## 9 — Report, package, and download

In [ ]:
!coderefine report
!coderefine export $ADAPTER --out-dir artifacts/release --base-model $BASE
from IPython.display import Markdown, display
display(Markdown(open("reports/EXPERIMENT_REPORT.md").read()))

In [ ]:
# Zip the adapter + all evaluation artifacts and pull them down
!zip -qr coderefine_results.zip artifacts/release artifacts/eval artifacts/runs artifacts/forgetting reports data/processed/dataset_card.json data/benchmark/benchmark_card.json
!du -h coderefine_results.zip
from google.colab import files
files.download("coderefine_results.zip")

## 10 — Try the A/B server here

Serves base and fine-tuned from one resident model by toggling the adapter.

In [ ]:
import subprocess, time, requests, json
proc = subprocess.Popen(
    ["coderefine","serve","--base-model",BASE,"--adapter",ADAPTER,"--load-in-4bit","--port","8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _ in range(90):
    try:
        if requests.get("http://127.0.0.1:8000/healthz", timeout=2).ok: break
    except Exception: time.sleep(2)
print(requests.get("http://127.0.0.1:8000/healthz").json())

payload = {
 "lang":"py",
 "old_code":"def load_config(path):\n    try:\n        with open(path) as fh:\n            return json.load(fh)\n    except Exception:\n        return None",
 "comment":"This except block swallows the error and returns None, which hides real failures. Just let it propagate.",
 "gold":"def load_config(path):\n    with open(path) as fh:\n        return json.load(fh)",
}
print(json.dumps(requests.post("http://127.0.0.1:8000/ab", json=payload, timeout=300).json(), indent=2))
# proc.terminate()

In [ ]:
import sys, platform, os
print("python:", sys.version)
print("platform:", platform.platform())
try:
    import torch
    print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
        print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1))
except ImportError:
    print("torch not installed yet")
print("cwd:", os.getcwd())
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU visible"

python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
platform: Linux-6.6.122+-x86_64-with-glibc2.35
torch: 2.11.0+cu128 | cuda available: True
gpu: Tesla T4
vram_gb: 15.6
cwd: /content
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
import os
os.listdir('/content')

['.config', 'sample_data']

In [ ]:
import os
if os.path.exists('/content/upload.b64'):
    os.remove('/content/upload.b64')
os.makedirs('/content/project', exist_ok=True)
print("ready")

In [ ]:
import os
if os.path.exists('/content/upload.b64'):
    os.remove('/content/upload.b64')
os.makedirs('/content/project', exist_ok=True)
print("ready")

ready


In [ ]:
%cd /content
!rm -rf lora-code-refinement
!git clone -q https://github.com/hamnaraeel/lora-code-refinement.git
%cd lora-code-refinement
!ls -la
print("--- data ---")
!ls -la data/processed data/benchmark

/content
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'lora-code-refinement'
/content
total 20
drwxr-xr-x 1 root root 4096 Sep  4 13:56 .
drwxr-xr-x 1 root root 4096 Sep  4 13:35 ..
drwxr-xr-x 4 root root 4096 Aug 24 13:27 .config
drwxr-xr-x 2 root root 4096 Sep  4 13:42 project
drwxr-xr-x 1 root root 4096 Aug 24 13:28 sample_data
--- data ---
ls: cannot access 'data/processed': No such file or directory
ls: cannot access 'data/benchmark': No such file or directory


In [ ]:
%cd /content
!rm -rf lora-code-refinement
!git clone -q https://github.com/hamnaraeel/lora-code-refinement.git
%cd lora-code-refinement
!ls -la
print("--- data ---")
!ls -la data/processed data/benchmark

In [13]:
print("kernel responsive")
import os
print("cwd:", os.getcwd())

kernel responsive
cwd: /content


In [14]:
%cd /content
!rm -rf lora-code-refinement
!git clone -q https://github.com/hamnaraeel/lora-code-refinement.git
%cd lora-code-refinement
!ls -la
print("--- data ---")
!ls -la data/processed data/benchmark

/content
/content/lora-code-refinement
total 76
drwxr-xr-x 10 root root  4096 Sep  4 14:05 .
drwxr-xr-x  1 root root  4096 Sep  4 14:05 ..
drwxr-xr-x  3 root root  4096 Sep  4 14:05 configs
drwxr-xr-x  4 root root  4096 Sep  4 14:05 data
drwxr-xr-x  2 root root  4096 Sep  4 14:05 docker
-rw-r--r--  1 root root   891 Sep  4 14:05 .env.example
drwxr-xr-x  8 root root  4096 Sep  4 14:05 .git
-rw-r--r--  1 root root   923 Sep  4 14:05 .gitignore
-rw-r--r--  1 root root  2963 Sep  4 14:05 Makefile
drwxr-xr-x  2 root root  4096 Sep  4 14:05 notebooks
-rw-r--r--  1 root root   632 Sep  4 14:05 pyproject.toml
-rw-r--r--  1 root root 12202 Sep  4 14:05 README.md
-rw-r--r--  1 root root   807 Sep  4 14:05 requirements-gpu.txt
-rw-r--r--  1 root root  1220 Sep  4 14:05 requirements.txt
drwxr-xr-x  2 root root  4096 Sep  4 14:05 scripts
drwxr-xr-x  3 root root  4096 Sep  4 14:05 src
drwxr-xr-x  2 root root  4096 Sep  4 14:05 tests
--- data ---
data/benchmark:
total 64
drwxr-xr-x 2 root root  4096 

In [ ]:
%cd /content/lora-code-refinement
!pip install -q -U "transformers>=4.44" "peft>=0.12" "trl>=0.9.6" "accelerate>=0.33" "datasets>=2.19" "bitsandbytes>=0.43.1" sacrebleu Levenshtein pyyaml pydantic typer rich matplotlib 2>&1 | tail -15
!pip install -q -e . --no-deps 2>&1 | tail -5
import torch, transformers, peft, trl, bitsandbytes
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)
print("bitsandbytes", bitsandbytes.__version__)

In [15]:
%cd /content/lora-code-refinement
!pip install -q -U "transformers>=4.44" "peft>=0.12" "trl>=0.9.6" "accelerate>=0.33" "datasets>=2.19" "bitsandbytes>=0.43.1" sacrebleu Levenshtein pyyaml pydantic typer rich matplotlib 2>&1 | tail -15
!pip install -q -e . --no-deps 2>&1 | tail -5
import torch, transformers, peft, trl, bitsandbytes
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)
print("bitsandbytes", bitsandbytes.__version__)

/content/lora-code-refinement
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 135.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.9 MB/s eta 0:00:00
ERROR: pip

In [2]:
import os, subprocess
print("cwd:", os.getcwd())
print(subprocess.run(["ls","-la"], capture_output=True, text=True).stdout)


cwd: /content
total 16
drwxr-xr-x 1 root root 4096 Aug 24 13:28 .
drwxr-xr-x 1 root root 4096 Sep  4 16:09 ..
drwxr-xr-x 4 root root 4096 Aug 24 13:27 .config
drwxr-xr-x 1 root root 4096 Aug 24 13:28 sample_data



In [3]:
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv"], capture_output=True, text=True).stdout)
print(subprocess.run(["find","/content","-maxdepth","2"], capture_output=True, text=True).stdout)


name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07

/content
/content/.config
/content/.config/.last_survey_prompt.yaml
/content/.config/configurations
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/.last_opt_in_prompt.yaml
/content/.config/active_config
/content/.config/config_sentinel
/content/.config/logs
/content/.config/default_configs.db
/content/.config/.last_update_check.json
/content/.config/gce
/content/sample_data
/content/sample_data/README.md
/content/sample_data/anscombe.json
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_train_small.csv



In [4]:
import subprocess
r = subprocess.run(["git","clone","-q","https://github.com/hamnaraeel/loRA-code-refinement.git"], capture_output=True, text=True, cwd="/content")
print("stdout:", r.stdout)
print("stderr:", r.stderr)
print("returncode:", r.returncode)


stdout: 
stderr: 
returncode: 0


In [5]:
import subprocess
r = subprocess.run(["ls","-la","data/processed","data/benchmark"], capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print(r.stdout, r.stderr)


data/benchmark:
total 64
drwxr-xr-x 2 root root  4096 Sep  4 16:14 .
drwxr-xr-x 4 root root  4096 Sep  4 16:14 ..
-rw-r--r-- 1 root root  1140 Sep  4 16:14 benchmark_card.json
-rw-r--r-- 1 root root 37593 Sep  4 16:14 benchmark.jsonl
-rw-r--r-- 1 root root  9564 Sep  4 16:14 handwritten.jsonl

data/processed:
total 2412
drwxr-xr-x 2 root root    4096 Sep  4 16:14 .
drwxr-xr-x 4 root root    4096 Sep  4 16:14 ..
-rw-r--r-- 1 root root    6814 Sep  4 16:14 dataset_card.json
-rw-r--r-- 1 root root  239272 Sep  4 16:14 test.jsonl
-rw-r--r-- 1 root root 1964921 Sep  4 16:14 train.jsonl
-rw-r--r-- 1 root root  245030 Sep  4 16:14 valid.jsonl
 


In [6]:
import subprocess
cmd = ("pip install -q -U 'transformers>=4.44' 'peft>=0.12' 'trl>=0.9.6' 'accelerate>=0.33' "
       "'datasets>=2.19' 'bitsandbytes>=0.43.1' sacrebleu Levenshtein pyyaml pydantic typer rich matplotlib")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print(r.stdout[-3000:])
print("STDERR TAIL:", r.stderr[-2000:])
print("rc:", r.returncode)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
import subprocess
r = subprocess.run("pip install -q -e . --no-deps", shell=True, capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print(r.stdout[-2000:], r.stderr[-2000:], r.returncode)

r2 = subprocess.run("python -c \"import torch, transformers, peft, trl, bitsandbytes; print('torch', torch.__version__, '| cuda', torch.cuda.is_available()); print('transformers', transformers.__version__, '| peft', peft.__version__, '| trl', trl.__version__)\"", shell=True, capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print(r2.stdout, r2.stderr)


  0
torch 2.11.0+cu128 | cuda True
transformers 5.16.1 | peft 0.20.0 | trl 1.12.0
 


In [8]:
import subprocess
cmd = ('pip install -q "transformers==4.46.3" "peft==0.13.2" "trl==0.11.4" "accelerate==1.2.1" "tokenizers<0.21" '
       '"bitsandbytes>=0.43.1" "datasets>=2.19" sacrebleu Levenshtein pyyaml pydantic typer rich matplotlib wandb')
r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print(r.stdout[-3000:])
print("STDERR TAIL:", r.stderr[-3000:])
print("rc:", r.returncode)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 25.5 MB/s eta 0:00:00

STDERR TAIL: ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, bu

In [9]:
import subprocess
r = subprocess.run("pip install -q -e . --no-deps", shell=True, capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print("editable install rc:", r.returncode, r.stderr[-1000:])

code = '''
import sys
if "/content/loRA-code-refinement/src" not in sys.path:
    sys.path.insert(0, "/content/loRA-code-refinement/src")
import torch, transformers, peft, trl, accelerate, bitsandbytes
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0), "| vram_gb:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__, "| accelerate", accelerate.__version__)
print("bitsandbytes", bitsandbytes.__version__)
from trl import DataCollatorForCompletionOnlyLM
print("legacy masking path: OK")
'''
r2 = subprocess.run(["python","-c",code], capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print(r2.stdout)
print("STDERR:", r2.stderr[-3000:])


editable install rc: 0 
torch 2.11.0+cu128 | cuda True
gpu: Tesla T4 | vram_gb: 15.6
transformers 4.46.3 | peft 0.13.2 | trl 0.11.4 | accelerate 1.2.1
bitsandbytes 0.50.2
legacy masking path: OK

STDERR: 2026-09-04 16:16:45.665439: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



In [10]:
import subprocess, os
env = os.environ.copy()
env["PYTHONPATH"] = "/content/loRA-code-refinement/src"
proc = subprocess.Popen(
    ["coderefine", "train", "configs/qlora_mistral7b.yaml"],
    cwd="/content/loRA-code-refinement",
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
print("started training, pid:", proc.pid)


started training, pid: 2126


In [11]:
import time
time.sleep(20)
# non-blocking read of what's available so far
import fcntl, os
fd = proc.stdout.fileno()
fl = fcntl.fcntl(fd, fcntl.F_GETFL)
fcntl.fcntl(fd, fcntl.F_SETFL, fl | os.O_NONBLOCK)
try:
    data = proc.stdout.read()
except Exception as e:
    data = f"(read error: {e})"
print("poll status:", proc.poll())
print(data)


poll status: None
2026-09-04 16:17:05.425870: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[train] TRL masking mode: legacy DataCollatorForCompletionOnlyLM
[train] sequence lengths: {'seq_len_mean': 263.0, 'seq_len_p50': 250, 'seq_len_p95': 422, 'seq_len_max': 802, 'seq_len_truncated_pct': 0.0}



In [12]:
import time
time.sleep(90)
data = proc.stdout.read()
print("poll status:", proc.poll())
print(data[-3000:] if data else "(no new output)")


poll status: None



In [13]:
import time
time.sleep(100)
data = proc.stdout.read()
print("poll status:", proc.poll())
print(data[-3000:] if data else "(no new output)")


poll status: None


Loading checkpoint shards:  33%|███▎      | 1/3 [00:20<00:41, 20.60s/it]


In [14]:
import time
time.sleep(60)
data = proc.stdout.read()
print("poll status:", proc.poll())
print(data[-3000:] if data else "(no new output)")


poll status: None
xception e) {
				indexProject.exception(e, "Failed to index artifact %s", pom.binaryArchive());
			}
		}
```</s></s></s></s></s></s></s></s></s></s></s></s></s> This instance will be ignored in loss calculation. Note, if this happens often, consider increasing the `max_seq_length`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/trl/trainer/utils.py:156: UserWarning: Could not find response key `[/INST] ` in the following instance: <s><s>[INST] Language: Go

Code under review:
```go
			}
			callbacks := network.AllocationCallbacks{
				GetReservedNetworks: r.getReservedSubnets,
				PersistAllocatedNetwork: func(ctx context.Context, subnet net.IPNet) error {
					return r.splitAndPersistReservedSubnet(ctx, cr, subnet, randomAZs)
				},
			}
			subnetCIDR, err = r.networkAllocator.Allocate(ctx, r.networkRange, r.allocatedSubnetMask, callbacks)
```

Reviewer comment:
It would make sense to me to have something like this. ```go callbacks := network.AllocationCal

In [15]:
print("poll status:", proc.poll())
data = proc.stdout.read()
print(data[-3000:] if data else "(no new output)")


poll status: None
me_import_allowed_extensions'],
                ],
        ]);
    }
}
```</s> This instance will be ignored in loss calculation. Note, if this happens often, consider increasing the `max_seq_length`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/trl/trainer/utils.py:156: UserWarning: Could not find response key `[/INST] ` in the following instance: <s><s>[INST] Language: Go

Code under review:
```go
func transactionPayloadAttrsNotInFields() *tests.Set {
	return tests.NewSet(
		tests.Group("transaction.marks."),
		tests.Group("context.db"),
		"transaction.span_count.started",
	)
}
```

Reviewer comment:
Please remove `tests.Group("context.db")`, it only concerns `spans`.

Revised code:[/INST] ```go
func transactionPayloadAttrsNotInFields() *tests.Set {
	return tests.NewSet(
		tests.Group("transaction.marks."),
		"transaction.span_count.started",
	)
}
```</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s><

In [16]:
import time
time.sleep(30)
new_data = proc.stdout.read()
full = (data or "") + (new_data or "")
print("poll status:", proc.poll())
warn_count = full.count("Could not find response key")
print("warning count so far (this read window):", warn_count)
# look for actual training step/loss lines
import re
loss_lines = [l for l in full.splitlines() if re.search(r"'loss':|it/s\]$|train_runtime|^\{.*loss", l)]
print("last 10 potential progress/loss lines:")
for l in loss_lines[-10:]:
    print(l)


TypeError: can't concat NoneType to bytes

In [17]:
import os, time
def nb_read():
    try:
        chunk = os.read(proc.stdout.fileno(), 65536)
        return chunk.decode(errors="replace")
    except BlockingIOError:
        return ""
    except Exception as e:
        return f"(read err: {e})"

time.sleep(5)
new_data = nb_read()
full = (full or "") + new_data
print("poll status:", proc.poll())
print("total captured chars:", len(full))
warn_count = full.count("Could not find response key")
print("warning count in captured buffer:", warn_count)
import re
loss_lines = [l for l in full.splitlines() if "'loss'" in l or re.search(r"^\s*\d+%\|.*it/s\]", l) or "train_runtime" in l]
print("last progress/loss lines:")
for l in loss_lines[-15:]:
    print(l)


NameError: name 'full' is not defined

In [18]:
import os, time
def nb_read():
    try:
        chunk = os.read(proc.stdout.fileno(), 200000)
        return chunk.decode(errors="replace")
    except BlockingIOError:
        return ""
    except Exception as e:
        return f"(read err: {e})"

time.sleep(5)
new_data = nb_read()
full = (data or "") + new_data
print("poll status:", proc.poll())
print("total captured chars:", len(full))
warn_count = full.count("Could not find response key")
print("warning count in captured buffer:", warn_count)
import re
loss_lines = [l for l in full.splitlines() if "'loss'" in l or re.search(r"^\s*\d+%\|.*it/s\]", l) or "train_runtime" in l]
print("last progress/loss lines:")
for l in loss_lines[-15:]:
    print(l)


poll status: None
total captured chars: 17855
warning count in captured buffer: 16
last progress/loss lines:


In [19]:
import subprocess
code = '''
import sys, json
sys.path.insert(0, "/content/loRA-code-refinement/src")
from transformers import AutoTokenizer
from coderefine.train import find_response_template
import coderefine.prompts as prompts

tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
template = find_response_template(tok)
print("response_template repr:", repr(template))
template_ids = tok.encode(template, add_special_tokens=False)
print("template_ids:", template_ids, [tok.decode([i]) for i in template_ids])

rows = [json.loads(l) for l in open("data/processed/train.jsonl")]
print("total examples:", len(rows))

def find_sublist(big, small):
    n, m = len(big), len(small)
    for i in range(n - m + 1):
        if big[i:i+m] == small:
            return i
    return -1

miss = 0
for r in rows:
    msgs = prompts.build_training_messages(r) if hasattr(prompts, "build_training_messages") else None
    text = tok.apply_chat_template(msgs, tokenize=False) if msgs else None
    if text is None:
        continue
    ids = tok.encode(text, add_special_tokens=False)
    if find_sublist(ids, template_ids) == -1:
        miss += 1
print("checked:", len(rows), "| token-level miss:", miss, f"({100*miss/len(rows):.1f}%)")
'''
r = subprocess.run(["python","-c",code], capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print(r.stdout[-3000:])
print("STDERR:", r.stderr[-2000:])


response_template repr: '[/INST] '
template_ids: [4, 29473] ['[/INST]', '']
total examples: 2000

STDERR: 2026-09-04 16:28:47.063983: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "<string>", line 26, in <module>
    msgs = prompts.build_training_messages(r) if hasattr(prompts, "build_training_messages") else None
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^
TypeError: build_training_messages() missing 3 required positional arguments: 'comment', 'new_code', and 'lang'



In [20]:
import subprocess
code = '''
import sys, json
sys.path.insert(0, "/content/loRA-code-refinement/src")
from transformers import AutoTokenizer
from coderefine.train import find_response_template, render_dataset

tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
template = find_response_template(tok)
template_ids = tok.encode(template, add_special_tokens=False)
print("response_template repr:", repr(template), "ids:", template_ids)

rows = [json.loads(l) for l in open("data/processed/train.jsonl")]
ds = render_dataset(rows, tok)
print("dataset size:", len(ds))
print("dataset columns:", ds.column_names)

def find_sublist(big, small):
    n, m = len(big), len(small)
    for i in range(n - m + 1):
        if big[i:i+m] == small:
            return i
    return -1

text_field = "text" if "text" in ds.column_names else ds.column_names[0]
miss = 0
miss_examples = []
for i, row in enumerate(ds):
    text = row[text_field]
    ids = tok.encode(text, add_special_tokens=False)
    if find_sublist(ids, template_ids) == -1:
        miss += 1
        if len(miss_examples) < 2:
            miss_examples.append(text[-300:])
print("checked:", len(ds), "| token-level miss:", miss, f"({100*miss/len(ds):.1f}%)")
for ex in miss_examples:
    print("---MISS EXAMPLE TAIL---")
    print(ex)
'''
r = subprocess.run(["python","-c",code], capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print(r.stdout[-4000:])
print("STDERR:", r.stderr[-2500:])


response_template repr: '[/INST] ' ids: [4, 29473]
dataset size: 2000
dataset columns: ['text', 'id', 'lang', 'category']
checked: 2000 | token-level miss: 2000 (100.0%)
---MISS EXAMPLE TAIL---
Identity(),SPHERE,sphere_params,"default",false);
  model->addElement(3,0,Matrix4d::Identity(),SPHERE,sphere_params,"default",false);
  model->updateElementsForBody(1, T_body1_to_world);
  model->updateElementsForBody(2, T_body2_to_world);
  model->updateElementsForBody(3, T_body3_to_world);
```</s>
---MISS EXAMPLE TAIL---
lectors.toList());
        //add hideMessageIds to log header, only for default logging, since for binary logging, the messages will be only hidden in console.log.
        //This is printed when its configured in bootstrap.properties
        if (hideMessageids.size() > 0 && !isHpelEnabled) {
```</s>

STDERR: 


In [21]:
import subprocess
code = '''
import sys, json
sys.path.insert(0, "/content/loRA-code-refinement/src")
from transformers import AutoTokenizer
from coderefine.train import render_dataset

tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
rows = [json.loads(l) for l in open("data/processed/train.jsonl")][:1]
ds = render_dataset(rows, tok)
text = ds[0]["text"]
print("RAW TEXT (first 200 chars):", repr(text[:200]))
print()
ids_no_special = tok.encode(text, add_special_tokens=False)
ids_with_special = tok(text)["input_ids"]
print("id4 (INST close token) present in no_special encode:", 4 in ids_no_special)
print("id4 present in default encode:", 4 in ids_with_special)
print("count of id4 in no_special:", ids_no_special.count(4))
print("first 20 ids no_special:", ids_no_special[:20])
print([tok.decode([i]) for i in ids_no_special[:20]])
'''
r = subprocess.run(["python","-c",code], capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print(r.stdout)
print("STDERR:", r.stderr[-2000:])


RAW TEXT (first 200 chars): '<s>[INST] Language: C++\n\nCode under review:\n```cpp\n  T_body3_to_world.topRightCorner(3,1) << 2,2,0;\n  shared_ptr<Model> model = newModel();\n  model->addElementWithoutGroup(1,0,Matrix4d::Identity(),BOX'

id4 (INST close token) present in no_special encode: True
id4 present in default encode: True
count of id4 in no_special: 1
first 20 ids no_special: [1, 3, 16357, 29515, 1102, 2448, 781, 781, 3308, 1684, 4826, 29515, 781, 14708, 29600, 5990, 781, 29473, 1088, 29498]
['<s>', '[INST]', 'Language', ':', 'C', '++', '\n', '\n', 'Code', 'under', 'review', ':', '\n', '``', '`', 'cpp', '\n', '', 'T', '_']

STDERR: 2026-09-04 16:30:01.745599: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



In [22]:
print("training proc still running:", proc.poll() is None)


training proc still running: True


In [23]:
import sys, json
sys.path.insert(0, "/content/loRA-code-refinement/src")
from transformers import AutoTokenizer
from coderefine.train import render_dataset

tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
rows = [json.loads(l) for l in open("/content/loRA-code-refinement/data/processed/train.jsonl")][:1]
ds = render_dataset(rows, tok)
text = ds[0]["text"]
print("RAW TEXT (first 200 chars):", repr(text[:200]))
ids_no_special = tok.encode(text, add_special_tokens=False)
print("id4 present in no_special encode:", 4 in ids_no_special, "count:", ids_no_special.count(4))
print("first 20 ids:", ids_no_special[:20])
print([tok.decode([i]) for i in ids_no_special[:20]])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


RAW TEXT (first 200 chars): '<s>[INST] Language: C++\n\nCode under review:\n```cpp\n  T_body3_to_world.topRightCorner(3,1) << 2,2,0;\n  shared_ptr<Model> model = newModel();\n  model->addElementWithoutGroup(1,0,Matrix4d::Identity(),BOX'
id4 present in no_special encode: True count: 1
first 20 ids: [1, 3, 16357, 29515, 1102, 2448, 781, 781, 3308, 1684, 4826, 29515, 781, 14708, 29600, 5990, 781, 29473, 1088, 29498]
['<s>', '[INST]', 'Language', ':', 'C', '++', '\n', '\n', 'Code', 'under', 'review', ':', '\n', '``', '`', 'cpp', '\n', '', 'T', '_']


In [24]:
idx = ids_no_special.index(4)
print("idx of [/INST]:", idx, "total len:", len(ids_no_special))
print("tokens around it:", ids_no_special[idx-2:idx+6])
print([tok.decode([i]) for i in ids_no_special[idx-2:idx+6]])
print("does [4, 29473] appear as consecutive pair:", ids_no_special[idx:idx+2])


idx of [/INST]: 262 total len: 455
tokens around it: [3464, 29515, 4, 9557, 5990, 781, 29473, 1088]
['code', ':', '[/INST]', '```', 'cpp', '\n', '', 'T']
does [4, 29473] appear as consecutive pair: [4, 9557]


In [25]:
print("killing broken training run (near-zero learning signal due to masking bug)")
proc.terminate()
import time
time.sleep(3)
print("poll after terminate:", proc.poll())
if proc.poll() is None:
    proc.kill()
    time.sleep(2)
    print("poll after kill:", proc.poll())


killing broken training run (near-zero learning signal due to masking bug)
poll after terminate: -15


In [26]:
fixed_source = r'''PLACEHOLDER'''
print("placeholder cell ready")


placeholder cell ready


In [27]:
path = "/content/loRA-code-refinement/src/coderefine/train.py"
src = open(path).read()

old1 = '''    prefix = with_answer[:marker]
    # Walk back to just after the user content to isolate the assistant header.
    user_end = prefix.rfind("U")
    template = prefix[user_end + 1 :] if user_end != -1 else prefix
    if not template.strip():
        raise RuntimeError("Empty response template derived from chat template.")
    return template'''

new1 = '''    prefix = with_answer[:marker]
    # Walk back to just after the user content to isolate the assistant header.
    user_end = prefix.rfind("U")
    template = prefix[user_end + 1 :] if user_end != -1 else prefix
    # Trailing whitespace in the template (e.g. the space in "[/INST] ") is not
    # a stable token boundary: SentencePiece merges it into the first token of
    # whatever follows (real targets start with a code fence, so "[/INST] ```"
    # tokenizes with that space fused into a "\\u2581```" token, not as a standalone
    # space token). Matching on that fused token-id sequence then never occurs
    # in real data, so the collator silently fails to find the response key on
    # (in practice) every example. Stripping trailing whitespace keeps the
    # match anchored to the structural token(s) that are always split out on
    # their own, and lets the fused leading-space+content token land inside
    # the (correctly loss-visible) completion instead of the template.
    template = template.rstrip()
    if not template:
        raise RuntimeError("Empty response template derived from chat template.")
    return template'''

assert old1 in src, "old1 not found"
src = src.replace(old1, new1, 1)

old2 = '''    collator = None
    if use_legacy_collator:
        collator = DataCollatorForCompletionOnlyLM(
            response_template=find_response_template(tokenizer), tokenizer=tokenizer
        )'''
new2 = '''    collator = None
    if use_legacy_collator:
        response_template = find_response_template(tokenizer)
        collator = DataCollatorForCompletionOnlyLM(
            response_template=response_template, tokenizer=tokenizer
        )
        _check_response_template_coverage(tokenizer, train_ds, response_template)'''
assert old2 in src, "old2 not found"
src = src.replace(old2, new2, 1)

marker = '''# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------'''
assert src.count(marker) == 1
helper = '''def _check_response_template_coverage(tokenizer, dataset, response_template: str, sample_size: int = 200) -> None:
    """Fail loudly if the response template's token ids don't actually occur in real examples."""
    template_ids = tokenizer.encode(response_template, add_special_tokens=False)
    text_field = "text" if "text" in dataset.column_names else dataset.column_names[0]
    n = min(sample_size, len(dataset))
    misses = 0
    for i in range(n):
        ids = tokenizer.encode(dataset[i][text_field], add_special_tokens=False)
        found = any(ids[j : j + len(template_ids)] == template_ids for j in range(len(ids) - len(template_ids) + 1))
        if not found:
            misses += 1
    miss_rate = misses / n
    print(f"[train] response template coverage: {n - misses}/{n} examples matched ({100 * (1 - miss_rate):.1f}%)")
    if miss_rate > 0.02:
        raise RuntimeError(
            f"Response template {response_template!r} (ids={template_ids}) was not found in "
            f"{misses}/{n} sampled training examples ({100 * miss_rate:.1f}% miss rate). "
            "Completion-only loss masking would silently no-op (or near-no-op) on this data."
        )


''' + marker
src = src.replace(marker, helper, 1)

open(path, "w").write(src)
print("patched OK, new file size:", len(src))


patched OK, new file size: 26315


In [28]:
import subprocess
code = '''
import sys, json
sys.path.insert(0, "/content/loRA-code-refinement/src")
from transformers import AutoTokenizer
from coderefine.train import find_response_template, render_dataset, _check_response_template_coverage

tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
template = find_response_template(tok)
print("response_template repr:", repr(template))

rows = [json.loads(l) for l in open("data/processed/train.jsonl")]
ds = render_dataset(rows, tok)
_check_response_template_coverage(tok, ds, template, sample_size=len(ds))
print("PASSED")
'''
r = subprocess.run(["python","-c",code], capture_output=True, text=True, cwd="/content/loRA-code-refinement")
print(r.stdout[-1500:])
print("STDERR:", r.stderr[-1500:])
print("rc:", r.returncode)


response_template repr: '[/INST]'
[train] response template coverage: 2000/2000 examples matched (100.0%)
PASSED

STDERR: 
rc: 0


In [29]:
import subprocess, os
env = os.environ.copy()
env["PYTHONPATH"] = "/content/loRA-code-refinement/src"
proc = subprocess.Popen(
    ["coderefine", "train", "configs/qlora_mistral7b.yaml"],
    cwd="/content/loRA-code-refinement",
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
print("started training, pid:", proc.pid)


started training, pid: 6788


In [30]:
import os, time
def nb_read():
    try:
        chunk = os.read(proc.stdout.fileno(), 200000)
        return chunk.decode(errors="replace")
    except BlockingIOError:
        return ""
    except Exception as e:
        return f"(read err: {e})"

time.sleep(45)
out = nb_read()
print("poll:", proc.poll())
print(out[-3000:])


poll: None
[train] TRL masking mode: legacy DataCollatorForCompletionOnlyLM
[train] sequence lengths: {'seq_len_mean': 263.0, 'seq_len_p50': 250, 'seq_len_p95': 422, 'seq_len_max': 802, 'seq_len_truncated_pct': 0.0}
Loading checkpoint shards:  33%|███▎      | 1/3 [00:19<00:39, 19.98s/it]


In [31]:
import time
time.sleep(60)
out = nb_read()
print("poll:", proc.poll())
print(out[-4000:])


poll: None
Loading checkpoint shards: 100%|██████████| 3/3 [00:58<00:00, 19.64s/it]
[train] trainable 6,815,744 / 3,765,178,368 (0.181%)
[train] response template coverage: 200/200 examples matched (100.0%)
[train] note: SFTConfig does not accept ['assistant_only_loss'] in this version; not set.
Map: 100%|██████████| 250/250 [00:00<00:00, 2123.62 examples/s]
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
  0%|          | 0/375 [00:00<?, ?it/s]


In [32]:
import time
time.sleep(90)
out = nb_read()
print("poll:", proc.poll())
print(out[-3000:])


poll: None
  0%|          | 1/375 [01:28<9:14:40, 88.99s/it]


In [33]:
import time
time.sleep(90)
out = nb_read()
print("poll:", proc.poll())
print(out[-3000:])


poll: None
  1%|          | 2/375 [03:01<9:24:29, 90.80s/it]


In [34]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


Fri Sep  4 16:41:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             61W /   70W |    6173MiB /  15360MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [35]:
proc.terminate()
import time
time.sleep(3)
print("poll:", proc.poll())
import subprocess
r = subprocess.run("pip install -q bitsandbytes==0.45.0", shell=True, capture_output=True, text=True)
print(r.stdout[-1000:], r.stderr[-1500:], r.returncode)


poll: -15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 11.8 MB/s eta 0:00:00
  0


In [36]:
import subprocess, os
env = os.environ.copy()
env["PYTHONPATH"] = "/content/loRA-code-refinement/src"
proc2 = subprocess.Popen(
    ["coderefine", "train", "configs/qlora_mistral7b.yaml", "--set", "train.max_train_samples=32", "name=speedtest"],
    cwd="/content/loRA-code-refinement",
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
print("started speed test, pid:", proc2.pid)


started speed test, pid: 8672


In [37]:
import os, time
def nb_read2():
    try:
        chunk = os.read(proc2.stdout.fileno(), 200000)
        return chunk.decode(errors="replace")
    except BlockingIOError:
        return ""
    except Exception as e:
        return f"(read err: {e})"

time.sleep(75)
out = nb_read2()
print("poll:", proc2.poll())
print(out[-3000:])


poll: 2
Usage: coderefine train [OPTIONS] {config}
Try 'coderefine train --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Got unexpected extra argument(s) (name=speedtest)                            │
╰──────────────────────────────────────────────────────────────────────────────╯



In [38]:
import subprocess, os
env = os.environ.copy()
env["PYTHONPATH"] = "/content/loRA-code-refinement/src"
proc2 = subprocess.Popen(
    ["coderefine", "train", "configs/qlora_mistral7b.yaml", "--set", "train.max_train_samples=32", "--set", "name=speedtest"],
    cwd="/content/loRA-code-refinement",
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
print("started speed test, pid:", proc2.pid)
import time
time.sleep(80)
out = nb_read2()
print("poll:", proc2.poll())
print(out[-2500:])


started speed test, pid: 9067
poll: 1
 │                                                                  │
│    81 │   │   bnb_multibackend_is_enabled =                                  │
│       is_bitsandbytes_multi_backend_available()                              │
│ in _handle_fromlist:1420                                                     │
│                                                                              │
│ /usr/local/lib/python3.13/dist-packages/transformers/utils/import_utils.py:1 │
│ 766 in __getattr__                                                           │
│                                                                              │
│   1763 │   │   │                                                             │
│   1764 │   │   │   value = Placeholder                                       │
│   1765 │   │   elif name in self._class_to_module.keys():                    │
│ ❱ 1766 │   │   │   module = self._get_module(self._class_to_module[name])    │
│

In [ ]:
import subprocess
r = subprocess.run("pip install -q bitsandbytes==0.43.1", shell=True, capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-1500:], r.returncode)


In [39]:
import subprocess
r = subprocess.run('pip install -q "bitsandbytes>=0.43.1"', shell=True, capture_output=True, text=True)
print("restored, rc:", r.returncode)


restored, rc: 0
